In [1]:
# !pip install pandas 
# !pip install openpyxl 
# !pip install chromadb 
# !pip install sentence-transformers
# !pip install boto3
# !pip install gradio
# !pip install ipywidgets


In [ ]:
import pandas as pd
import re, json, uuid, csv, os
from datetime import datetime
from collections import Counter
import boto3
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder

# ── CONFIG ──────────────────────────────────────────────────────────────────
FILEPATH         = 'Updated_DataLake_Questionnaire.xlsx'    # Source Excel file with all 3 sheets
CHROMA_DIR       = './mbp_rag_db_advanced'                  # Local folder where ChromaDB persists embeddings
COLLECTION_NAME  = 'mbp_rag_collection_v2'                  # Name of the ChromaDB collection
EMBED_MODEL      = 'all-MiniLM-L6-v2'                       # Lightweight but accurate sentence embedding model
RERANK_MODEL     = 'cross-encoder/ms-marco-MiniLM-L-6-v2'   # Cross-encoder used to rerank retrieved chunks
BEDROCK_REGION   = 'ap-south-1'                             # AWS region where Bedrock is available
BEDROCK_MODEL_ID = 'anthropic.claude-3-haiku-20240307-v1:0' # Claude 3 Haiku — fast + accurate LLM on Bedrock
TOP_K_RETRIEVE   = 5                                        # How many chunks to pull from ChromaDB during semantic search
TOP_K_RERANK     = 3                                        # How many of those to keep after reranking (best 3 out of 5)
LOG_FILE         = 'chat_log.csv'                           # Every Q&A gets logged here for audit/debugging

print('Config loaded.')


2026-03-04 20:11:29.612467: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Config loaded.


In [3]:
def normalize_key(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r'[^a-z0-9_]', '_', text)
    return re.sub(r'_+', '_', text).strip('_')

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    def clean(col):
        col = str(col).strip().lower().replace(' ', '_')
        col = re.sub(r'[^a-z0-9_]', '_', col)
        return re.sub(r'_+', '_', col).strip('_')
    df.columns = [clean(c) for c in df.columns]
    return df

def build_standardized_file_key(filename: str):
    if pd.isna(filename): return None
    fn = str(filename).lower().strip()
    fn = re.sub(r'\.txt|\.csv', '', fn)
    fn = fn.replace('-', '_')
    fn = re.sub(r'\d{8}', '', fn)
    fn = re.sub(r'yyyy_mm_dd|yyyymmdd', '', fn)
    return re.sub(r'_+', '_', fn).strip('_')

print('Helpers ready.')


Helpers ready.


In [ ]:
# ── VERIFY SHEET NAMES BEFORE LOADING ──
import pandas as pd
xl = pd.ExcelFile(FILEPATH)
print('Sheets found in your Excel file:')
for s in xl.sheet_names:
    print(f'  → {s}')

# Expected sheet names used
SHEET_QUESTIONNAIRE  = 'Questionnaire'
SHEET_FILE_INFO      = 'File Information if File Based'
SHEET_MAPPING        = 'Metadata_OR_Mapping'

# Validate
missing = [s for s in [SHEET_QUESTIONNAIRE, SHEET_FILE_INFO, SHEET_MAPPING]
           if s not in xl.sheet_names]
if missing:
    print(f'\n MISSING SHEETS: {missing}')
    print('Update SHEET_QUESTIONNAIRE / SHEET_FILE_INFO / SHEET_MAPPING above to match your actual sheet names.')
else:
    print('\n All 3 sheets found. Safe to proceed.')


Sheets found in your Excel file:
  → Questionnaire
  → File Information if File Based
  → Metadata_OR_Mapping

✅ All 3 sheets found. Safe to proceed.


In [ ]:
# ============================================================
# SHEET LOADERS: READ AND CLEAN ALL 3 SHEETS
# ============================================================
# Each function handles one sheet. They all follow the same pattern:
#   1. Read raw Excel with no header assumption (header=None)
#   2. Drop fully empty rows
#   3. Dynamically detect the real header row by scanning for known keywords
#   4. Set that row as column names and slice data below it
#   5. Normalize column names to snake_case
#   6. Return the clean DataFrame


def load_questionnaire_sheet(filepath):
    """
    Loads Sheet 1 (Questionnaire) and splits it into two parts:
    - project_df     : the top 5 rows with project-level metadata (name, ID, impact score, etc.)
    - questionnaire_df : the main questionnaire responses (category, description, response)

    Why split?
    The top rows are used to build the global_governance_context dictionary (control layer).
    The bottom rows are chunked for semantic search in ChromaDB.
    """
    # Read raw without assuming any header row
    df = pd.read_excel(filepath, sheet_name='Questionnaire', header=None)
    df = df.dropna(how='all').reset_index(drop=True)

    # Lowercase all cells to safely detect the header row
    dflower = df.apply(lambda col: col.astype(str).str.strip().str.lower())

    # Find the row that contains "field", "description", "value" — that's our real header
    header_mask = dflower.apply(lambda r: {'field','description','value'}.issubset(set(r)), axis=1)
    if not header_mask.any():
        raise ValueError('Header row not found in Questionnaire')
    header_idx = header_mask.idxmax()

    # Set that row as column names, then take everything below it as data
    df.columns = df.iloc[header_idx].astype(str).str.strip().str.lower()
    df = df.iloc[header_idx+1:].reset_index(drop=True)

    # The questionnaire section starts where field == "category"
    # Everything above that line is project metadata; everything below is questionnaire data
    category_rows = df[df['field'].astype(str).str.strip().str.lower() == 'category']
    split_idx = category_rows.index[0]

    project_df        = df.iloc[:split_idx].copy().reset_index(drop=True)
    questionnaire_df  = df.iloc[split_idx+1:].copy().reset_index(drop=True)
    questionnaire_df.columns = ['category', 'description', 'response']

    # Strip whitespace from string cells
    questionnaire_df = questionnaire_df.apply(
        lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x))
    return project_df, questionnaire_df


def load_file_information_sheet(filepath):
    """
    Loads Sheet 2 (File Information if File Based).
    Each row = one physical file delivered by the source system.
    Contains ingestion metadata: frequency, transmission type, schema, mailboxes, etc.

    Also adds a 'standardized_file_key' column — the stable join key used to
    link this sheet with Sheet 3 (field mappings).
    """
    raw = pd.read_excel(filepath, sheet_name='File Information if File Based', header=None)
    raw = raw.dropna(how='all').reset_index(drop=True)

    # Find the real header row by looking for the "file name" keyword
    dflower = raw.apply(lambda col: col.astype(str).str.strip().str.lower())
    header_mask = dflower.apply(lambda r: 'file name' in list(r), axis=1)
    if not header_mask.any():
        raise ValueError('Header not found in File Information')
    header_idx = header_mask.idxmax()

    df = raw.iloc[header_idx:].reset_index(drop=True)
    df.columns = df.iloc[0]           # first row becomes column headers
    df = df.iloc[1:].reset_index(drop=True)
    df = normalize_columns(df)        # snake_case all columns

    # Build the stable join key from the raw filename
    df['standardized_file_key'] = df['file_name'].apply(build_standardized_file_key)
    return df.reset_index(drop=True)


def load_metadata_mapping_sheet(filepath, file_information_df):
    """
    Loads Sheet 3 (Metadata_OR_Mapping).
    Each row = one field in a target table, linked to a specific file and record type.
    Contains: actual_file_name, record_type, schema_name, table_name, column_name, field_type, max_length, etc.

    Also adds 'standardized_file_key' so this sheet can be joined with Sheet 2.
    Note: No cross-validation with Sheet 2 here — that is done implicitly via file_tree construction.
    """
    raw = pd.read_excel(filepath, sheet_name='Metadata_OR_Mapping', header=None)
    raw = raw.dropna(how='all').reset_index(drop=True)

    # Find header row by scanning for "actual file name"
    dflower = raw.apply(lambda col: col.astype(str).str.strip().str.lower())
    header_mask = dflower.apply(lambda r: 'actual file name' in list(r), axis=1)
    if not header_mask.any():
        raise ValueError('Header not found in Metadata_OR_Mapping')
    header_idx = header_mask.idxmax()

    df = raw.iloc[header_idx:].reset_index(drop=True)
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)
    df = normalize_columns(df)

    # Build the stable join key to link back to Sheet 2
    df['standardized_file_key'] = df['actual_file_name'].apply(build_standardized_file_key)
    return df.reset_index(drop=True)


# Load all sheets
project_df, questionnaire_df = load_questionnaire_sheet(FILEPATH)
file_information_df           = load_file_information_sheet(FILEPATH)
metadata_mapping_df           = load_metadata_mapping_sheet(FILEPATH, file_information_df)

print(f'Sheet1 — project: {project_df.shape}, questionnaire: {questionnaire_df.shape}')
print(f'Sheet2 — file info: {file_information_df.shape}')
print(f'Sheet3 — mapping: {metadata_mapping_df.shape}')


Sheet1 — project: (5, 3), questionnaire: (52, 3)
Sheet2 — file info: (13, 26)
Sheet3 — mapping: (64, 14)


In [ ]:
# ============================================================
# BUILDING GLOBAL GOVERNANCE CONTEXT 
# ============================================================
# Extracts a flat key-value dictionary from both Sheet 1 sections.
# This is NOT chunked or embedded. It is used as:
#   - Metadata injected into every chunk (so the LLM always knows the project context)
#   - Direct lookup for deterministic governance questions
#   - Routing signal for domain detection


def extract_project_metadata(project_df):
    """
    Reads the top 5 rows of Sheet 1 (project-level fields like name, ID, impact score).
    
    """
    metadata = {}
    for _, row in project_df.iterrows():
        key = normalize_key(str(row['field']))
        value = row['value']
        if pd.notna(value):
            metadata[key] = str(value).strip()
    return metadata

def extract_governance_attributes(questionnaire_df):
    """
    Scans questionnaire rows for specific category/description signals and
    extracts structured governance attributes needed for routing and filtering.

    Why pattern-match instead of taking everything?
    We only want the critical control-layer fields here (source name, security class,
    environments, approval status, etc.). All other questionnaire content will be
    chunked and embedded separately for semantic search.
    """
    gov = {}
    for _, row in questionnaire_df.iterrows():
        cat  = str(row['category']).strip().lower()
        desc = str(row['description']).strip().lower()
        resp = row['response']
        if pd.isna(resp): continue
        resp = str(resp).strip()
        if cat == 'source name':                                         gov['source_name']             = resp
        if cat == 'business line':                                       gov['business_line']           = resp
        if cat == 'data access' and 'host type' in desc:                gov['host_type']               = resp
        if cat == 'data access' and 'delivery method' in desc:          gov['delivery_method']         = resp
        if cat == 'availability' and 'how often' in desc:               gov['data_frequency']          = resp
        if cat == 'file db information' and 'security' in desc.lower(): gov['security_classification'] = resp
        if cat == 'environments':                                        gov['environments']            = resp
        if cat == 'data retention period':                               gov['data_retention_period']   = resp
        if cat == 'source owner approval':                               gov['source_owner_approval']   = resp
    return gov

project_metadata          = extract_project_metadata(project_df)
governance_attributes     = extract_governance_attributes(questionnaire_df)
global_governance_context = {**project_metadata, **governance_attributes}

print('Global Governance Context:')
for k, v in global_governance_context.items():
    print(f'  {k}: {v}')


Global Governance Context:
  project_name: MBP 3.11 Upgrade
  planview_project_id: 8679
  impact_score: 90
  strategic_initiative_program_name: EDP
  enactment_pod_name: Ancillary Integration
  source_name: FIS Profile
  business_line: Consumer
  host_type: External
  delivery_method: GIS
  data_frequency: Daily
  environments: PROD,QA
  data_retention_period: Default 7 year
  source_owner_approval: Pending approval from Source Owner


In [ ]:
# ============================================================
# BUILDING FILE TREE (IN-MEMORY HIERARCHY)
# ============================================================
# Merges Sheet 2 (file-level metadata) with Sheet 3 (field-level mappings)
# into a nested dictionary structure. This tree lives only in memory during chunking.
# It is NOT stored in ChromaDB. Its sole purpose is to organize data for chunk generation.
#
# Structure:
# file_tree = {
#   "mbp_accountextract_4300": {
#     "file_info": { ...row from Sheet 2... },
#     "mappings": {
#       "9000": {
#         "APD_AR_TDL_DP": [ {field row 1}, {field row 2}, ... ]
#       }
#     }
#   },
#   ...
# }

def build_file_tree(file_information_df, metadata_mapping_df):
    """
    Step 1: Initialize one node per file from Sheet 2
    Step 2: For each row in Sheet 3, find the matching file node and nest the
            field under its record_type -> table_name path.
    If a Sheet 3 row has a key that doesn't exist in Sheet 2, it is skipped.
    """
    file_tree = {}
    # Initialize file nodes from Sheet 2
    for _, row in file_information_df.iterrows():
        key = row['standardized_file_key']
        file_tree[key] = {'file_info': row.to_dict(), 'mappings': {}}

    # Populate field mappings from Sheet 3 under the matching file node
    for _, row in metadata_mapping_df.iterrows():
        key         = row['standardized_file_key']
        record_type = str(row.get('record_type', 'UNKNOWN'))
        table_name  = row.get('table_name', 'UNKNOWN')

        if key not in file_tree:
            continue  # Sheet 3 has a file not in Sheet 2 — skip gracefully

        # Create nested dict layers if they don't exist yet
        if record_type not in file_tree[key]['mappings']:
            file_tree[key]['mappings'][record_type] = {}
        if table_name not in file_tree[key]['mappings'][record_type]:
            file_tree[key]['mappings'][record_type][table_name] = []

        # Append this field row under file → record_type → table
        file_tree[key]['mappings'][record_type][table_name].append(row.to_dict())

    return file_tree

file_tree = build_file_tree(file_information_df, metadata_mapping_df)
print(f'File tree: {len(file_tree)} files')
print(f'Keys: {list(file_tree.keys())}')


File tree: 13 files
Keys: ['mbp_accountextract_4300', 'mbp_arrangement_extract_delta_4300', 'mbp_ext_att_ar_instance_extract_delta_4300', 'mbp_ext_att_ar_instance_extract_delta_totals_4300', 'mbp_ext_att_def_extract_delta_4300', 'mbp_ext_att_def_extract_delta_totals_4300', 'mbp_ext_att_ip_instance_extract_delta_4300', 'mbp_ext_att_ip_instance_extract_delta_totals_4300', 'mbp_involved_party_extract_delta_4300', 'mbp_involved_party_extract_delta_totals_4300', 'mbp_ip_ar_rltnp_extract_delta_4300', 'mbp_ip_ip_rltnp_extract_delta_4300', 'mbp_transactionextract_4300']


In [ ]:
# ============================================================
# BUILD ALL CHUNKS (GOVERNANCE + FILE + TECHNICAL)
# ============================================================
# We create 3 types of chunks — each type goes into the same ChromaDB collection
# but is tagged with a 'domain' metadata field so the router can filter them.
#
# Chunk count breakdown:
#   - Governance chunks  : 1 project overview + 1 per questionnaire category = 26
#   - File info chunks   : 1 per physical file = 13
#   - Technical chunks   : 1 per (file × record_type × table) combination = 13
#   - TOTAL = 52 chunks




# ── GOVERNANCE CHUNKS (Sheet 1) ──
def build_governance_chunks(project_df, questionnaire_df, global_governance_context):
    """
    Creates two types of governance chunks:
    1. One project_overview chunk — all key-value pairs from the top project metadata rows
    2. One chunk per questionnaire category — groups all Q&A rows under that category

    Each chunk gets 'domain': 'governance' in metadata so the router can filter to
    only governance chunks when answering questions about vendor, risk, retention, etc.
    """
    chunks = []
    # Chunk 1: Project overview — all project-level fields as "Field: Value" lines
    project_lines = [
        f"{row['field']}: {row['value']}"
        for _, row in project_df.iterrows() if pd.notna(row['value'])
    ]
    chunks.append({
        'content': '\n'.join(project_lines),
        'metadata': {
            'domain': 'governance', 'section': 'project_overview',
            'source_name':   global_governance_context.get('source_name', ''),
            'project_name':  global_governance_context.get('project_name', '')
        }
    })

    # Chunk 2..N: One chunk per questionnaire category
    for category, group in questionnaire_df.groupby('category'):
        lines = [
            f"{row['description']}: {row['response']}"
            for _, row in group.iterrows() if pd.notna(row['response'])
        ]
        if not lines: continue  # skip empty categories

        content = f"Category: {category}\n" + '\n'.join(lines)
        chunks.append({
            'content': content,
            'metadata': {
                'domain': 'governance', 'section': 'questionnaire',
                'category':     category.strip().lower(),
                'source_name':  global_governance_context.get('source_name', ''),
                'project_name': global_governance_context.get('project_name', '')
            }
        })
    return chunks


# ── FILE INFORMATION CHUNKS (Sheet 2) ──
def build_file_information_chunks(file_information_df):
    chunks = []
    for _, row in file_information_df.iterrows():
        lines = [
            f"File Name: {row.get('file_name')}",
            f"Description: {row.get('file_description_include_subject_area')}",
            f"Transmission Type: {row.get('transmission_type_gis_ndm_other')}",
            f"Frequency: {row.get('frequency_daily_weekly_monthly')}",
            f"Load Type: {row.get('full_delta')}",
            f"Autosys Start: {row.get('autosys_start')}",
            f"Schema: {row.get('datalake_schema_name_raw_cl')}",
            f"Encrypted: {row.get('encrypted_y_n')}",
            f"Compressed: {row.get('compressed_y_n')}",
            f"Producer Mailbox: {row.get('original_producer_mail_box')}",
            f"Consumer Mailbox: {row.get('consumer_mail_box')}",
            f"File Size: {row.get('file_size_bytes_')}",
            f"Holiday Rules: {row.get('holiday_rules')}",
            f"In Scope Acquisition: {row.get('in_scope_for_acquisition_y_n_if_n_then_reason')}",
            f"In Scope Ingestion: {row.get('in_scope_for_ingestion_y_n_if_n_then_reason')}",
        ]
        content = '\n'.join([l for l in lines if 'None' not in str(l)])
        metadata = {
            'domain':       'file_information',
            'file_key':     str(row.get('standardized_file_key', '')),
            'frequency':    str(row.get('frequency_daily_weekly_monthly', '')),
            'load_type':    str(row.get('full_delta', '')),
            'transmission': str(row.get('transmission_type_gis_ndm_other', '')),
            'schema':       str(row.get('datalake_schema_name_raw_cl', ''))
        }
        chunks.append({'content': content, 'metadata': metadata})
    return chunks


# ── TECHNICAL MAPPING CHUNKS (Sheet 3) ──
def build_technical_chunks(file_tree):
    chunks = []
    for file_key, file_data in file_tree.items():
        file_info = file_data['file_info']
        mappings  = file_data['mappings']
        for record_type, tables in mappings.items():
            for table_name, fields in tables.items():
                lines = [
                    f"File: {file_info.get('file_name')}",
                    f"Record Type: {record_type}",
                    f"Target Schema: {fields[0].get('schema_name') if fields else ''}",
                    f"Target Table: {table_name}",
                    f"Frequency: {file_info.get('frequency_daily_weekly_monthly')}",
                    f"Transmission: {file_info.get('transmission_type_gis_ndm_other')}",
                    f"Load Type: {file_info.get('full_delta')}",
                    '\nFields:'
                ]
                for f in fields:
                    lines.append(
                        f"  - {f.get('field_name')} ({f.get('field_type')}, "
                        f"max={f.get('max_length')}): "
                        f"{f.get('_field_description', f.get('field_description', ''))}"
                    )
                content  = '\n'.join(lines)
                metadata = {
                    'domain':       'technical_mapping',
                    'file_key':     file_key,
                    'record_type':  str(record_type),
                    'schema':       str(fields[0].get('schema_name', '')) if fields else '',
                    'table':        str(table_name),
                    'frequency':    str(file_info.get('frequency_daily_weekly_monthly', '')),
                    'load_type':    str(file_info.get('full_delta', '')),
                    'transmission': str(file_info.get('transmission_type_gis_ndm_other', ''))
                }
                chunks.append({'content': content, 'metadata': metadata})
    return chunks


governance_chunks = build_governance_chunks(project_df, questionnaire_df, global_governance_context)
file_chunks       = build_file_information_chunks(file_information_df)
technical_chunks  = build_technical_chunks(file_tree)
all_chunks        = governance_chunks + file_chunks + technical_chunks

print(f'Governance: {len(governance_chunks)}, File: {len(file_chunks)}, Technical: {len(technical_chunks)}')
print(f'Total chunks: {len(all_chunks)}')


Governance: 26, File: 13, Technical: 13
Total chunks: 52


In [ ]:
#  ANALYTICAL LAYER (AGGREGATE FACTS) ────────────────────────
# semantic search are useful but they dont give you a clear answer so we are computing all aggregate facts that can be directly computed from the data
# this will prevent any kind of hallucination from the LLM
import pandas as pd
 
def build_analytical_facts(metadata_mapping_df, file_information_df):
    """
    Organized into three sections:
    A) Field-level analysis  (from Sheet 3 / Metadata_OR_Mapping)
    B) File scope & mailbox  (from Sheet 2 / File Information)
    C) Autosys timing        (from Sheet 2 / File Information)
    D) Cross-sheet join      (autosys hour → files + record types)
    """
    mdf = metadata_mapping_df.copy()
    fdf = file_information_df.copy()
 
    # Normalize column names to simple snake_case
    mdf.columns = [str(c).strip().lower().replace(' ', '_') for c in mdf.columns]
    fdf.columns = [str(c).strip().lower().replace(' ', '_') for c in fdf.columns]
 
    facts = {}
 
    # -------- Mapping sheet (Metadata_OR_Mapping) -------------------------
 
    # Field name, field type, max length, record type, actual file name
    fn_col = next((c for c in mdf.columns if c == 'field_name' or 'field_name' in c), None)
    ft_col = next((c for c in mdf.columns if 'field' in c and 'type' in c), None)
    ml_col = next((c for c in mdf.columns if 'max' in c and 'length' in c), None)
    rt_col = next((c for c in mdf.columns if 'record' in c and 'type' in c), None)
    af_col = next((c for c in mdf.columns if 'actual' in c and 'file' in c), None)
 
    # 1) Total fields
    facts['total_fields'] = len(mdf)
 
    # 2) Field type distribution + per-type field names
    if ft_col:
        ft_series = mdf[ft_col]
        if isinstance(ft_series, pd.DataFrame):
            ft_series = ft_series.iloc[:, 0]
        ft_series = ft_series.astype(str).str.strip().str.upper()
        type_counts = ft_series.value_counts().to_dict()
        facts['field_type_distribution'] = type_counts
        for ftype, count in type_counts.items():
            key = ftype.lower()
            facts[f'total_{key}_fields'] = count
            if fn_col:
                facts[f'{key}_field_names'] = (
                    mdf.loc[ft_series == ftype, fn_col].astype(str).tolist()
                )
 
    # 3) Fields with max length > 40
    if ml_col and fn_col and ft_col:
        ml_series = mdf[ml_col]
        if isinstance(ml_series, pd.DataFrame):
            ml_series = ml_series.iloc[:, 0]
        mdf['_ml_num'] = pd.to_numeric(ml_series, errors='coerce')
        big_fields = mdf[mdf['_ml_num'] > 40][[fn_col, ft_col, ml_col]].dropna()
        facts['fields_with_max_length_gt_40'] = big_fields.to_dict(orient='records')
        facts['fields_with_max_length_gt_40_count'] = len(big_fields)
        facts['fields_with_max_length_gt_40_names'] = big_fields[fn_col].astype(str).tolist()
 
    # 4) Unique record types and field counts per record type
    if rt_col:
        rt_series = mdf[rt_col]
        if isinstance(rt_series, pd.DataFrame):
            rt_series = rt_series.iloc[:, 0]
        rt_series = rt_series.astype(str).str.strip()
        unique_rt = sorted(rt_series.unique().tolist())
        facts['unique_record_types'] = unique_rt
        facts['unique_record_types_count'] = len(unique_rt)
        facts['fields_per_record_type'] = rt_series.value_counts().to_dict()
 
    # 5) Per-file field counts (Actual File Name)
    if af_col:
        af_series = mdf[af_col]
        if isinstance(af_series, pd.DataFrame):
            af_series = af_series.iloc[:, 0]
        af_series = af_series.astype(str).str.strip()
        facts['fields_per_file'] = af_series.value_counts().to_dict()
 
    # -------- File Information sheet (File Information if File Based) -----
 
    fn2_col  = next((c for c in fdf.columns if c == 'file_name'), None)
    acq_col  = next((c for c in fdf.columns if 'acquisition' in c), None)
    ing_col  = next((c for c in fdf.columns if 'ingestion' in c), None)
    prod_col = next((c for c in fdf.columns if 'producer' in c), None)
    cons_col = next((c for c in fdf.columns if 'consumer' in c and 'mailbox' in c), None)
    auto_col = next((c for c in fdf.columns if 'autosys' in c), None)
 
    # 6) Files in scope for acquisition AND ingestion
    if fn2_col and acq_col and ing_col:
        acq_series = fdf[acq_col].astype(str).str.strip().str.lower()
        ing_series = fdf[ing_col].astype(str).str.strip().str.lower()
        in_scope = fdf[acq_series.str.startswith('y') & ing_series.str.startswith('y')]
        facts['files_in_scope_acquisition_and_ingestion'] = (
            in_scope[fn2_col].astype(str).tolist()
        )
        facts['files_in_scope_count'] = len(in_scope)
 
    # 7) Producer / consumer mailbox per file + grouping by producer
    if fn2_col and prod_col:
        facts['producer_mailbox_per_file'] = dict(
            zip(fdf[fn2_col].astype(str), fdf[prod_col].astype(str))
        )
        facts['files_grouped_by_producer_mailbox'] = (
            fdf.groupby(prod_col)[fn2_col].apply(lambda s: s.astype(str).tolist()).to_dict()
        )
    if fn2_col and cons_col:
        facts['consumer_mailbox_per_file'] = dict(
            zip(fdf[fn2_col].astype(str), fdf[cons_col].astype(str))
        )
 
    # 8) Autosys start per file + files before 6 AM + autosys-hour groups
    if fn2_col and auto_col:
        fdf['__autosys'] = fdf[auto_col].astype(str)
 
        facts['autosys_start_per_file'] = dict(
            zip(fdf[fn2_col].astype(str), fdf['__autosys'])
        )
 
        def parse_hour(val: str) -> int:
            try:
                v = str(val).upper().replace('EST', '').strip()
                v = v.replace('AM', '').replace('PM', '').strip()
                hour = int(v.split(':')[0].strip())
                return hour
            except Exception:
                return 99
 
        fdf['__hour'] = fdf['__autosys'].apply(parse_hour)
        before6 = fdf[fdf['__hour'] < 6]
        facts['files_starting_before_6am'] = dict(
            zip(before6[fn2_col].astype(str), before6['__autosys'])
        )
        facts['files_starting_before_6am_count'] = len(before6)
 
        facts['files_per_autosys_hour'] = (
            fdf.groupby('__autosys')[fn2_col]
            .apply(lambda s: s.astype(str).tolist())
            .to_dict()
        )
 
    # 9) Join: autosys hour → (files, record types)
    if fn2_col and auto_col and af_col and rt_col:
        f_info = fdf[[fn2_col, '__autosys']].copy()
        f_info.columns = ['file_name_join', 'autosys_start']
        m_info = mdf[[af_col, rt_col]].drop_duplicates().copy()
        m_info.columns = ['file_name_join', 'record_type']
 
        f_info['file_name_join'] = f_info['file_name_join'].astype(str).str.strip()
        m_info['file_name_join'] = m_info['file_name_join'].astype(str).str.strip()
 
        merged = pd.merge(f_info, m_info, on='file_name_join', how='left')
        hour_rt = merged.groupby('autosys_start').apply(
            lambda x: {
                'files': x['file_name_join'].astype(str).tolist(),
                'record_types': sorted(x['record_type'].astype(str).unique().tolist())
            }
        ).to_dict()
        facts['files_and_record_types_by_autosys_hour'] = hour_rt
 
    return facts
 
 
analytical_facts = build_analytical_facts(metadata_mapping_df, file_information_df)
 
print("=== Analytical Facts summary ===")
for k, v in analytical_facts.items():
    print(f"{k}: {v}")
 

=== Analytical Facts summary ===
total_fields: 64
field_type_distribution: {'STRING': 47, 'NUMBER': 17}
total_string_fields: 47
string_field_names: ['ARID_ITEM_IBAN', 'ARID_ITEM', 'ARID_ITEM_MICR', 'ARID_ITEM', 'ARID_ITEM_MICR', 'ARID_ITEM_IBAN', 'SORT_APPL_VALUE', 'APA_EFCTV_DATE', 'PDELP_VALUE', 'PECT_VALUE', 'APA_FRST_EFCTV_DT', 'sorFeId', 'prcsrNme', 'applNme', 'fileNme', 'action_type', 'sortField1', 'sortField2', 'sortField3', 'sortField4', 'ARID_ITEM', 'ARID_ITEM_MICR', 'ARID_ITEM_IBAN', 'ARID_ITEM', 'ARID_ITEM_MICR', 'ARID_ITEM_IBAN', 'ARID_ITEM', 'ARID_ITEM_MICR', 'ARID_ITEM_IBAN', 'RsdncCtryNme', 'BrthDte', 'DthDte', 'FndtnDte', 'ARID_ITEM', 'ARID_ITEM_MICR', 'ARID_ITEM_IBAN', 'ARID_ITEM', 'ARID_ITEM_MICR', 'ARID_ITEM_IBAN', 'relApplNme', 'relSorIpId', 'strtDte', 'endDte', 'rltToCode', 'ARID_ITEM', 'ARID_ITEM_MICR', 'ARID_ITEM_IBAN']
total_number_fields: 17
number_field_names: ['REC_TYP', 'ARF_IDFR', 'REC_TYP', 'ARF_IDFR', 'feId', 'REC_TYP', 'ARF_IDFR', 'REC_TYP', 'ARF_IDFR', 

/tmp/ipykernel_2751/2198216955.py:49: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  facts['fields_with_max_length_gt_40'] = big_fields.to_dict(orient='records')
/tmp/ipykernel_2751/2198216955.py:145: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hour_rt = merged.groupby('autosys_start').apply(


In [ ]:
#VALIDATE ALL CHUNKS BEFORE INSERTING INTO CHROMADB
def validate_chunks(chunks):
    for i, chunk in enumerate(chunks):
        assert chunk.get('content'),            f'Chunk {i} has empty content'
        assert chunk.get('metadata'),           f'Chunk {i} missing metadata'
        assert 'domain' in chunk['metadata'],   f'Chunk {i} missing domain key'
    domain_dist = Counter(c['metadata']['domain'] for c in chunks)
    print(f'All {len(chunks)} chunks valid.')
    print(f'Domain distribution: {dict(domain_dist)}')

validate_chunks(all_chunks)


All 52 chunks valid.
Domain distribution: {'governance': 26, 'file_information': 13, 'technical_mapping': 13}


In [ ]:
#INITIALIZING EMBEDDING MODEL AND CHROMADB

# Load embedding model
embedding_model = SentenceTransformer(EMBED_MODEL)
print(f'Embedding model loaded: {EMBED_MODEL}')

# Initialize ChromaDB with version-safe approach
# (API changed between chromadb < 0.4.x and >= 0.4.x)
try:
    chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
    print('Using PersistentClient (chromadb >= 0.4.x)')
except AttributeError:
    from chromadb.config import Settings
    chroma_client = chromadb.Client(Settings(
        persist_directory=CHROMA_DIR,
        is_persistent=True
    ))
    print('Using legacy Client (chromadb < 0.4.x)')

# Delete old collection if it exists (fresh insert every run to avoid duplicates)
try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print('Deleted existing collection — fresh start.')
except:
    pass  # collection didn't exist yet, that's fine

collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)
print(f'Collection ready: {COLLECTION_NAME}')


Embedding model loaded: all-MiniLM-L6-v2
Using PersistentClient (chromadb >= 0.4.x)
Deleted existing collection — fresh start.
Collection ready: mbp_rag_collection_v2


In [ ]:
# ============================================================
# EMBED ALL CHUNKS AND INSERT INTO CHROMADB
# ============================================================
# Converts all 52 chunk texts into numerical vectors using the embedding model,
# then stores them in ChromaDB along with their metadata and a unique UUID per chunk.

documents = [c['content']  for c in all_chunks]  # text content of each chunk
metadatas = [c['metadata'] for c in all_chunks]  # domain + filter metadata per chunk
ids       = [str(uuid.uuid4()) for _ in all_chunks]  # unique ID required by ChromaDB

print('Generating embeddings — please wait...')
embeddings = embedding_model.encode(
    documents, convert_to_numpy=True, show_progress_bar=True
).tolist()

# Insert everything into ChromaDB in one batch
collection.add(
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings,
    ids=ids
)
print(f'Inserted {len(documents)} chunks into ChromaDB.')


Generating embeddings — please wait...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Inserted 52 chunks into ChromaDB.


In [ ]:
# Three things are detected:
# 1. Domain      — governance / file_information / technical_mapping
# 2. File key    — which specific file is being asked about (if any)
# 3. Record type — a 4-5 digit number like 9000, 9200 (if mentioned)
#
# These signals are combined into a ChromaDB metadata filter for precision retrieval.

available_file_keys = list(file_tree.keys())

# Keywords that strongly suggest a governance-domain question
GOVERNANCE_KEYWORDS = [
    'vendor', 'risk', 'contact', 'retention', 'business line', 'security',
    'approval', 'availability', 'environment', 'sourcing', 'strategy',
    'masking', 'dr', 'disaster recovery', 'financial', 'audience',
    'party data', 'documentation', 'change management', 'quality issue',
    'products', 'subject area', 'ad group'
]
# Keywords that suggest a field/column/mapping question (technical domain)
TECHNICAL_KEYWORDS = [
    'field', 'column', 'record type', 'datatype', 'table', 'schema',
    'mapping', 'target table', 'max length', 'column name'
]
# Keywords that suggest a file-level ingestion question
FILE_KEYWORDS = [
    'frequency', 'transmission', 'encrypted', 'compressed',
    'autosys', 'load type', 'mailbox', 'file size', 'holiday',
    'delta', 'full', 'producer', 'consumer', 'in scope'
]

def detect_domain(query: str) -> tuple:
    """
    Scores the query against each keyword list and picks the domain with the highest hit count.
    Returns (domain_name, confidence) where confidence = winning_score / total_score.
    Falls back to 'governance' with 0.0 confidence when no keywords match.
    """
    q = query.lower()
    gov_score  = sum(1 for w in GOVERNANCE_KEYWORDS  if w in q)
    tech_score = sum(1 for w in TECHNICAL_KEYWORDS   if w in q)
    file_score = sum(1 for w in FILE_KEYWORDS         if w in q)
    scores = {'governance': gov_score, 'technical_mapping': tech_score, 'file_information': file_score}
    best_domain = max(scores, key=scores.get)
    best_score  = scores[best_domain]
    if best_score == 0:
        return 'governance', 0.0
    confidence = best_score / (sum(scores.values()) + 1e-9)
    return best_domain, round(confidence, 2)

def detect_file_key(query: str) -> str:
    """
    Tries to identify which specific file the user is asking about.
    Cleans the query and checks how many significant parts of each file key appear in it.
    Requires a score >= 2 (at least 2 matching key segments > 3 chars) to avoid false positives.
    Returns the best matching file key or None if no confident match.
    """
    query_clean = re.sub(r'[^a-z0-9]', '', query.lower())
    best_match, best_score = None, 0
    for key in available_file_keys:
        key_parts = key.split('_')
        score = sum(2 for part in key_parts if len(part) > 3 and part in query_clean)
        if score > best_score:
            best_score = score
            best_match = key
    return best_match if best_score >= 2 else None

def detect_record_type(query: str) -> str:
    match = re.search(r'\b(\d{4,5})\b', query)
    return match.group() if match else None

def build_chroma_filter(domain=None, file_key=None, record_type=None):
    conditions = []
    if domain:      conditions.append({'domain':      {'$eq': domain}})
    if file_key:    conditions.append({'file_key':    {'$eq': file_key}})
    if record_type: conditions.append({'record_type': {'$eq': record_type}})
    if not conditions: return None
    if len(conditions) == 1: return conditions[0]
    return {'$and': conditions}

print('Query router ready.')


Query router ready.


In [ ]:
# ============================================================
# RETRIEVAL + RERANKING PIPELINE
# ============================================================
# Two-stage retrieval:
# Stage 1 (Semantic search): Use embedding similarity to fetch top-5 candidate chunks from ChromaDB
# Stage 2 (Reranking):       Use a cross-encoder to re-score those 5 chunks and keep the best 3
try:
    reranker     = CrossEncoder(RERANK_MODEL)
    USE_RERANKER = True
    print(f'Reranker loaded: {RERANK_MODEL}')
except Exception as e:
    reranker     = None
    USE_RERANKER = False
    print(f'Reranker unavailable ({e}), using semantic-only top-k.')


def retrieve_chunks(query: str, metadata_filter=None, top_k=TOP_K_RETRIEVE):
    """
    Encodes the query into a vector, then runs a similarity search in ChromaDB.
    If a metadata_filter is provided, only chunks matching that filter are considered.
    Returns (documents_list, metadatas_list).
    """
    query_embedding = embedding_model.encode(query, convert_to_numpy=True).tolist()
    kwargs = {'query_embeddings': [query_embedding], 'n_results': top_k}
    if metadata_filter:
        kwargs['where'] = metadata_filter
    results = collection.query(**kwargs)
    return results['documents'][0], results['metadatas'][0]


def rerank_chunks(query, docs, metas, top_k=TOP_K_RERANK):
    if not USE_RERANKER or len(docs) <= top_k:
        return docs[:top_k], metas[:top_k]
    scores = reranker.predict([(query, doc) for doc in docs])
    ranked = sorted(zip(scores, docs, metas), reverse=True)
    return [d for _, d, _ in ranked[:top_k]], [m for _, _, m in ranked[:top_k]]


def route_and_retrieve(query: str):
    """
    Full routing + retrieval pipeline:
    1. Detect domain, file_key, record_type from query
    2. Build ChromaDB metadata filter
    3. Retrieve top-K chunks (with filter)
    4. Fallback to domain-only filter if < 2 results found (prevents empty context)
    5. Rerank and return final top-K chunks
    """
    domain, confidence = detect_domain(query)
    file_key           = detect_file_key(query)
    record_type        = detect_record_type(query)
    metadata_filter    = build_chroma_filter(domain=domain, file_key=file_key, record_type=record_type)
    docs, metas        = retrieve_chunks(query, metadata_filter=metadata_filter, top_k=TOP_K_RETRIEVE)
    # Fallback if too few results
    if len(docs) < 2:
        fallback_filter = build_chroma_filter(domain=domain)
        docs, metas     = retrieve_chunks(query, metadata_filter=fallback_filter, top_k=TOP_K_RETRIEVE)
    docs, metas = rerank_chunks(query, docs, metas, top_k=TOP_K_RERANK)
    return docs, metas, domain, confidence, file_key, record_type

print('Retrieval + reranking pipeline ready.')


Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2
Retrieval + reranking pipeline ready.


In [ ]:
# BEDROCK LLM + ANSWER GENERATION WITH ANALYTICAL FACTS
# Calls Amazon Bedrock (Claude 3 Haiku) to generate the final answer.
# Two types of context are injected into the prompt:
#   1. RAG context   — the top-3 reranked chunks (always)
#   2. Analytical facts — pre-computed aggregates (only for count/list questions)

import json
import boto3

bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=BEDROCK_REGION
)

# System prompt: defines the LLM's role and strict rules

SYSTEM_PROMPT = """You are a Data Lake Governance and Technical Metadata Specialist
for the MBP 3.11 Upgrade project (FIS Profile source).

STRICT RULES:
1. Answer ONLY using the provided CONTEXT and ANALYTICAL FACTS. Never use outside knowledge.
2. Do NOT infer or fabricate any missing details.
3. If the answer is not in context, respond exactly:
   "The requested information is not available in the provided Data Lake documentation."
4. Be professional and structured. Use bullet points when listing multiple items.
5. For file questions: always include file name, frequency, and schema when available.
6. For field questions: always include field name, data type, and max length when available.
7. For COUNT / HOW MANY / TOTAL questions: always use the PRE-COMPUTED ANALYTICAL FACTS.
"""
# Pre-built facts summary string — injected into the prompt when the query is count/aggregate type
# This is formatted once and reused across all queries, keeping token usage efficient
facts_summary = f"""
PRE-COMPUTED ANALYTICAL FACTS (computed from full Excel, not from RAG):

FIELD TYPE / FIELDS:
- Total fields in mapping sheet          : {analytical_facts.get('total_fields')}
- Field type distribution                : {analytical_facts.get('field_type_distribution')}
- Total NUMBER fields                    : {analytical_facts.get('total_number_fields')}
- Total STRING fields                    : {analytical_facts.get('total_string_fields')}
- Fields with max length > 40 (count)    : {analytical_facts.get('fields_with_max_length_gt_40_count')}
- Fields with max length > 40 (details)  : {analytical_facts.get('fields_with_max_length_gt_40')}
- Unique record types                    : {analytical_facts.get('unique_record_types')}
- Unique record types count              : {analytical_facts.get('unique_record_types_count')}
- Fields per record type                 : {analytical_facts.get('fields_per_record_type')}
- Fields per file                        : {analytical_facts.get('fields_per_file')}

FILE SCOPE / MAILBOX:
- Files in scope (acquisition+ingestion) : {analytical_facts.get('files_in_scope_count')}
- File names in scope                    : {analytical_facts.get('files_in_scope_acquisition_and_ingestion')}
- Producer mailbox per file              : {analytical_facts.get('producer_mailbox_per_file')}
- Consumer mailbox per file              : {analytical_facts.get('consumer_mailbox_per_file')}
- Files grouped by producer mailbox      : {analytical_facts.get('files_grouped_by_producer_mailbox')}

TIMING / AUTOSYS:
- Autosys start per file                 : {analytical_facts.get('autosys_start_per_file')}
- Files starting before 6 AM EST         : {analytical_facts.get('files_starting_before_6am')}
- Files starting before 6 AM count       : {analytical_facts.get('files_starting_before_6am_count')}
- Files grouped by autosys start time    : {analytical_facts.get('files_per_autosys_hour')}
- Files + record types by autosys time   : {analytical_facts.get('files_and_record_types_by_autosys_hour')}
"""
# Keywords that indicate an aggregate/count/list question — trigger facts injection

AGG_KEYWORDS = [
    "how many", "count", "total", "number of",
    "how much", "list all", "all fields", "field type",
    "max length", "start before", "starting before", "start at", "autosys",
    "producer mailbox", "consumer mailbox"
]


def generate_answer(query: str, docs: list, conversation_history: list = None) -> str:
    """
    Builds the full prompt and sends it to Claude 3 Haiku via Amazon Bedrock.

    Prompt structure:
    [system prompt] + [analytical facts if needed] + [RAG context chunks] + [user question]

    Conversation history (last 4 turns) is prepended to support follow-up questions.
    Temperature is set to 0.0 for deterministic, factual answers.
    """
    # Join all retrieved chunks into one context block
    context = "\n---\n".join(map(str, docs))

    # Only inject analytical facts for count/aggregate/list queries
    q_lower    = query.lower()
    use_facts  = any(k in q_lower for k in AGG_KEYWORDS)
    extra_context = facts_summary if use_facts else ""

    # Build message list with conversation history (sliding window of last 4 turns)
    messages = []
    if conversation_history:
        for turn in conversation_history[-4:]:
            messages.append({"role": turn["role"], "content": turn["content"]})

    # Assemble the full user prompt: system rules + optional facts + RAG context + question
    full_prompt = (
        f"{SYSTEM_PROMPT}\n\n"
        f"{extra_context}\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"USER QUESTION:\n{query}\n\n"
        f"Answer:"
    )
    messages.append({"role": "user", "content": full_prompt})

    # Build Bedrock request payload (Claude Messages API format)
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens":  800,
        "temperature": 0.0,   # zero temperature = deterministic, no creative variation
        "messages":    messages
    }

    # Call Bedrock and extract the text response
    resp = bedrock_runtime.invoke_model(
        modelId=BEDROCK_MODEL_ID,
        body=json.dumps(body)
    )
    return json.loads(resp["body"].read())["content"][0]["text"]


print("Bedrock client with analytical layer ready.")

Bedrock client with analytical layer ready.


In [ ]:
#CONVERSATION LOGGER
# Every Q&A interaction is appended to a CSV file for audit trail & debugging

def log_conversation(query, answer, domain, confidence, file_key, record_type):
    file_exists = os.path.exists(LOG_FILE)
    with open(LOG_FILE, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=[
            'timestamp', 'query', 'answer', 'domain', 'confidence', 'file_key', 'record_type'
        ])
        if not file_exists:
            writer.writeheader()
        writer.writerow({
            'timestamp':   datetime.now().isoformat(),
            'query':       query,
            'answer':      answer,
            'domain':      domain,
            'confidence':  confidence,
            'file_key':    file_key or '',
            'record_type': record_type or ''
        })

print(f'Logger ready -> {LOG_FILE}')


Logger ready -> chat_log.csv


In [ ]:
#  MAIN RAG PIPELINE FUNCTION
# ask_rag_system() is the single entry point for all queries.
# It ties together: routing → retrieval → reranking → generation → memory → logging.
#
# Conversation memory is a sliding window of 20 messages (10 turns).
# When it exceeds 20, the oldest pair is dropped to stay within context limits.

conversation_history = []

def ask_rag_system(query: str, verbose: bool = True) -> str:
    
    """
    Full advanced RAG pipeline — executes these steps in order:
    1. Route   → detect domain, file_key, record_type from the query
    2. Retrieve → semantic search + metadata filter in ChromaDB (top-5)
    3. Rerank  → cross-encoder re-scores and selects best 3 chunks
    4. Generate → Claude 3 Haiku generates a grounded answer via Bedrock
    5. Memory  → question + answer appended to conversation_history
    6. Log     → row written to chat_log.csv
    """
    docs, metas, domain, confidence, file_key, record_type = route_and_retrieve(query)

    if verbose:
        print(f'[Router]    domain={domain} | confidence={confidence} | '
              f'file_key={file_key} | record_type={record_type}')
        print(f'[Retrieval] {len(docs)} chunks after reranking')

    answer = generate_answer(query, docs, conversation_history)

    conversation_history.append({'role': 'user',      'content': query})
    conversation_history.append({'role': 'assistant', 'content': answer})
    if len(conversation_history) > 20:
        conversation_history.pop(0)
        conversation_history.pop(0)

    log_conversation(query, answer, domain, confidence, file_key, record_type)
    return answer


def reset_conversation():
    global conversation_history
    conversation_history = []
    print('Conversation history cleared.')

print('RAG pipeline ready. Use ask_rag_system("your question") to query.')


RAG pipeline ready. Use ask_rag_system("your question") to query.


In [ ]:
#TEST WITH SAMPLE QUESTIONS

test_questions = [
    'What is the frequency of MBP Account Extract?',                 # file_information domain
    'Who is the vendor for this source?',                            # governance domain
    'What fields are in record type 9000 of MBP Account Extract?',   # technical_mapping domain
    'What is the data retention period?',                            # governance domain
    'Is MBP Account Extract encrypted?',                             # file_information domain
    'What are the environments available?',                          # governance domain
    'What is the security classification of this source?',           # governance domain
    'What is the autosys start time for the arrangement file?',      # file_information domain
    'What is the target table for MBP Transaction Extract?',         # technical_mapping domain
    'What is the CEO salary?',                                       # OUT-OF-SCOPE — should return "not available" response
]

reset_conversation()
for q in test_questions:
    print(f'\n{"="*70}')
    print(f'Q: {q}')
    answer = ask_rag_system(q)
    print(f'A: {answer}')


Conversation history cleared.

Q: What is the frequency of MBP Account Extract?
[Router]    domain=file_information | confidence=1.0 | file_key=mbp_accountextract_4300 | record_type=None
[Retrieval] 3 chunks after reranking
A: The frequency of the MBP_AccountExtract_4300_YYYYMMDD.txt file is Daily.

Q: Who is the vendor for this source?
[Router]    domain=governance | confidence=1.0 | file_key=None | record_type=None
[Retrieval] 3 chunks after reranking
A: The vendor for this source is FIS.

Q: What fields are in record type 9000 of MBP Account Extract?
[Router]    domain=technical_mapping | confidence=1.0 | file_key=mbp_accountextract_4300 | record_type=9000
[Retrieval] 3 chunks after reranking
A: Based on the provided context, the fields in record type 9000 of the MBP_AccountExtract_4300_YYYYMMDD.txt file are:

- ARID_ITEM_IBAN (STRING, max=40): Arrangement Account IBAN Number
- REC_TYP (NUMBER, max=4): Record Type
- ARF_IDFR (NUMBER, max=15): Arrangement Identifier
- ARID_ITEM (STRI

In [19]:
import ipywidgets as widgets
from IPython.display import display, clear_output

SAMPLE_QUESTIONS = [
    'What is the frequency of MBP Account Extract?',
    'What is the transmission type of MBP Transaction Extract?',
    'What fields are in record type 9000 of MBP Account Extract?',
    'Who is the vendor?',
    'What is the data retention period?',
    'What environments exist for this source?',
    'What is the autosys start time of the arrangement file?',
    'What is the sourcing strategy?',
    'What is the security classification?',
    'What are the change management contacts?',
]

sample_area  = widgets.Textarea(
    value='\n'.join(SAMPLE_QUESTIONS),
    description='Samples:',
    layout=widgets.Layout(width='100%', height='180px')
)
question_box = widgets.Textarea(
    placeholder='Type your question here and click Ask...',
    description='Question:',
    layout=widgets.Layout(width='100%', height='80px')
)
ask_btn  = widgets.Button(description='Ask',           button_style='primary')
clear_btn= widgets.Button(description='Clear History', button_style='warning')
out_box  = widgets.Output()

def on_ask(b):
    with out_box:
        clear_output()
        q = question_box.value.strip()
        if not q:
            print('Please enter a question.')
            return
        print(f'Processing: {q}...')
        print(f'\nAnswer:\n{ask_rag_system(q)}')

def on_clear(b):
    with out_box:
        clear_output()
        reset_conversation()

ask_btn.on_click(on_ask)
clear_btn.on_click(on_clear)

display(
    widgets.HTML('<h3>MBP 3.11 Upgrade — Data Lake RAG Chatbot</h3>'),
    sample_area,
    question_box,
    widgets.HBox([ask_btn, clear_btn]),
    out_box
)


HTML(value='<h3>MBP 3.11 Upgrade — Data Lake RAG Chatbot</h3>')

Textarea(value='What is the frequency of MBP Account Extract?\nWhat is the transmission type of MBP Transactio…

Textarea(value='', description='Question:', layout=Layout(height='80px', width='100%'), placeholder='Type your…

Output()

In [ ]:
#UI to launch a Gradio chatbot interface for the RAG system.
#Any UI change can be done here only, the backend RAG pipeline remains untouched.
import gradio as gr


def rag_chat(message, history):
    """
    message: latest user text (string)
    history: list of previous turns (ignored for now; we keep state in RAG)
    """
    try:
        reply = ask_rag_system(message, verbose=False)
    except Exception as e:
        reply = f"Error while answering: {e}"
    return reply


css = """
.gradio-container { max-width: 1100px !important; margin: auto; }
#title-bar { text-align: center; padding: 12px 0 4px 0; }
#title-bar h1 { margin: 0; font-size: 1.4rem; }
#title-bar p { margin: 4px 0 0 0; font-size: 0.85rem; opacity: 0.8; }
"""


with gr.Blocks(css=css, title="MBP DataLake RAG Chatbot") as demo:
    gr.HTML(
        """
        <div id="title-bar">
          <h1>MBP 3.11 Data Lake Assistant</h1>
          <p>Ask about files, fields, record types, mailboxes, timings, and governance.</p>
        </div>
        """
    )

    gr.ChatInterface(
        fn=rag_chat,
        chatbot=gr.Chatbot(
            label="MBP RAG Chat",
            height=520,
        ),
        textbox=gr.Textbox(
            placeholder=(
                "Ask a question, e.g. "
                "'Which files have record type 9200 and what are their consumer mailboxes?'"
            ),
            show_label=False,
            autofocus=True,
        ),
        examples=[
            "What is the autosys start time for MBP Transaction Extract?",
            "How many fields are NUMBER type across all files?",
            "Which files start before 6 AM EST?",
            "List all fields in record type 11000 with their max length.",
        ],
        cache_examples=False,
        submit_btn="Ask",
        # retry_btn="Retry",
        # undo_btn="Undo",
        # clear_btn="Clear",
    )


demo.launch(share=True)  # or share=False if only local

/tmp/ipykernel_2751/2903234379.py:24: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=css, title="MBP DataLake RAG Chatbot") as demo:
/tmp/ipykernel_2751/2903234379.py:34: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  gr.ChatInterface(


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://1ef35c02c46325219e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [21]:
if os.path.exists(LOG_FILE):
    log_df = pd.read_csv(LOG_FILE)
    print(f'Total logged queries: {len(log_df)}')
    display(log_df[['timestamp', 'query', 'domain', 'confidence', 'file_key']].tail(10))
else:
    print('No log file yet. Run some queries first.')


Total logged queries: 167


,timestamp,query,domain,confidence,file_key
157,2026-03-04T20:11:41.627220,What is the frequency of MBP Account Extract?,file_information,1.0,mbp_accountextract_4300
158,2026-03-04T20:11:42.265070,Who is the vendor for this source?,governance,1.0,NaN
159,2026-03-04T20:11:43.867423,What fields are in record type 9000 of MBP Acc...,technical_mapping,1.0,mbp_accountextract_4300
160,2026-03-04T20:11:44.672946,What is the data retention period?,governance,1.0,NaN
161,2026-03-04T20:11:45.866597,Is MBP Account Extract encrypted?,file_information,1.0,mbp_accountextract_4300
162,2026-03-04T20:11:46.555017,What are the environments available?,governance,1.0,NaN
163,2026-03-04T20:11:47.235440,What is the security classification of this so...,governance,1.0,NaN
164,2026-03-04T20:11:48.260109,What is the autosys start time for the arrange...,file_information,1.0,mbp_arrangement_extract_delta_4300
165,2026-03-04T20:11:49.195226,What is the target table for MBP Transaction E...,technical_mapping,1.0,mbp_arrangement_extract_delta_4300
166,2026-03-04T20:11:49.817418,What is the CEO salary?,governance,0.0,NaN
